<a href="https://colab.research.google.com/github/vv2281/MAT-422/blob/main/HW3_3_VV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Section 3.3: Optimality Conditions, Convexity, Gradient Descent

3.3.1 Necessary and sufficient conditions of local minimizers
3.3.2 Convexity and global minimizers
3.3.3 Gradient descent

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize

rng = np.random.default_rng(422)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

###3.3.1 Necessary and Sufficient Conditions of Local Minimizers

f: Rⁿ → R is twice differentiable, with gradient ∇f(x) and Hessian H(x) (the symmetric matrix of second partial derivatives).

x* is a local minimizer if f(x*) ≤ f(x) for all x within some distance δ > 0 of x*.
x* is a global minimizer if f(x*) ≤ f(x) for all x.

Second-order Taylor expansion around x*: f(x* + h) ≈ f(x*) + ∇f(x*)ᵀh + ½ hᵀH(x*)h

The conditions follow from this expansion:

  *First-order necessary:   x* local min  ⇒  ∇f(x*) = 0
  *Second-order necessary:  x* local min  ⇒  ∇f(x*) = 0 and H(x*) positive semidefinite (all eigenvalues ≥ 0)
  *Second-order sufficient: ∇f(x*) = 0 and H(x*) positive definite (all eigenvalues > 0)  ⇒  x* strict local min

A point with ∇f(x) = 0 is a critical (stationary) point. Classify it by the eigenvalues of H:
-all > 0: local min
-all < 0: local max
-some > 0 and some < 0: saddle point
-any = 0 with the rest one sign: test is inconclusive

##Why ∇f(x*) = 0 is necessary: descent directions

If ∇f(x) ≠ 0, the direction d = −∇f(x) has directional derivative: ∇f(x)ᵀd = −‖∇f(x)‖² < 0; so f(x − t∇f(x)) < f(x) for small t > 0, and x cannot be a local min.

Example: f(x, y) = x² + 3y² at (1, 1).

In [ ]:
f = lambda v: v[0]**2 + 3 * v[1]**2
grad_f = lambda v: np.array([2 * v[0], 6 * v[1]])

x0 = np.array([1.0, 1.0])
g = grad_f(x0)
print("grad f(1,1) =", g)
print("directional derivative along -grad =", -g @ g)
for t in [0.1, 0.01, 0.001]:
    print(f"t = {t:<6} f(x0 - t*grad) = {f(x0 - t*g):.6f}   f(x0) = {f(x0)}")

##Example 1: Classifying critical points

  f(x, y) = x³ − 3x − y³ + 3y

  ∇f = (3x² − 3, −3y² + 3) = 0   →   x = ±1, y = ±1   (4 critical points)

  H(x, y) = [[6x, 0], [0, −6y]]

Next, the gradient is checked to be 0 at each point and the point is classified by the Hessian eigenvalues.

In [ ]:
f1 = lambda x, y: x**3 - 3*x - y**3 + 3*y
grad_f1 = lambda x, y: np.array([3*x**2 - 3, -3*y**2 + 3])
hess_f1 = lambda x, y: np.array([[6*x, 0], [0, -6*y]])

def classify(H, tol=1e-10):
    eig = np.linalg.eigvalsh(H)
    if np.all(eig > tol): return "local min"
    if np.all(eig < -tol): return "local max"
    if eig.min() < -tol and eig.max() > tol: return "saddle"
    return "inconclusive"

points = [(1, 1), (1, -1), (-1, 1), (-1, -1)]
for p in points:
    H = hess_f1(*p)
    print(f"{str(p):9s} grad = {grad_f1(*p)}  eigenvalues = {np.linalg.eigvalsh(H)}  ->  {classify(H)}")

xs = np.linspace(-2.2, 2.2, 300)
Xg, Yg = np.meshgrid(xs, xs)
plt.figure(figsize=(6, 5))
cs = plt.contour(Xg, Yg, f1(Xg, Yg), levels=30, cmap="viridis")
plt.colorbar(cs)
markers = {"local min": "go", "local max": "r^", "saddle": "kx"}
for p in points:
    c = classify(hess_f1(*p))
    plt.plot(*p, markers[c], ms=11, label=c)
h, l = plt.gca().get_legend_handles_labels()
plt.legend(dict(zip(l, h)).values(), dict(zip(l, h)).keys())
plt.xlabel("x"); plt.ylabel("y"); plt.title("f = x³ − 3x − y³ + 3y")
plt.show()

##Example 2: The conditions are not equivalences

All three functions have f'(0) = 0 and f''(0) = 0 at x = 0, so the second-order necessary condition holds and the sufficient condition fails for all three.

  f(x) = x⁴   strict local min  (sufficient condition fails, but it is still a min)
  f(x) = x³   not a min or max  (necessary conditions hold, but it is not a min)
  f(x) = −x⁴  local max         (necessary conditions hold, but it is not a min)

So the necessary conditions can hold at a non-minimizer, and a minimizer can fail the sufficient condition.

In [ ]:
xs = np.linspace(-1.5, 1.5, 300)
funcs = {"x⁴": lambda x: x**4, "x³": lambda x: x**3, "−x⁴": lambda x: -x**4}

fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
for a, (name, fn) in zip(ax, funcs.items()):
    a.plot(xs, fn(xs))
    a.plot(0, 0, "ro")
    a.set_title(f"f(x) = {name}")
plt.tight_layout(); plt.show()

###Example 3: Numerical gradient and Hessian (Rosenbrock function)

When derivatives are hard to compute by hand, use central differences: ∂f/∂xᵢ ≈ [f(x + h eᵢ) − f(x − h eᵢ)] / (2h)

Rosenbrock function: f(x, y) = (1 − x)² + 100(y − x²)²

It has one critical point, (1, 1). The numerical Hessian there is positive definite, so (1, 1) is a strict local min.

In [ ]:
def num_grad(f, x, h=1e-5):
    x = np.asarray(x, float)
    return np.array([(f(x + h*e) - f(x - h*e)) / (2*h) for e in np.eye(len(x))])

def num_hess(f, x, h=1e-4):
    x = np.asarray(x, float)
    return np.array([(num_grad(f, x + h*e) - num_grad(f, x - h*e)) / (2*h) for e in np.eye(len(x))])

rosen = lambda v: (1 - v[0])**2 + 100 * (v[1] - v[0]**2)**2
rosen_hess = lambda v: np.array([[1200*v[0]**2 - 400*v[1] + 2, -400*v[0]], [-400*v[0], 200]])

xstar = np.array([1.0, 1.0])
H_num = num_hess(rosen, xstar)
print("numerical grad at (1,1):", np.round(num_grad(rosen, xstar), 6))
print("numerical Hessian:\n", np.round(H_num, 3))
print("exact Hessian:\n", rosen_hess(xstar))
print("eigenvalues:", np.round(np.linalg.eigvalsh(H_num), 3), "->", classify(H_num))

#3.3.2 Convexity and Global Minimizers

Convex set C: for x, y ∈ C and t ∈ [0, 1], tx + (1 − t)y ∈ C.

Convex function (on a convex set): the chord lies above the graph, f(tx + (1 − t)y) ≤ t f(x) + (1 − t) f(y)   for all x, y and t ∈ [0, 1]

Strictly convex: strict inequality whenever x ≠ y and 0 < t < 1.

Equivalent tests for differentiable f: First-order:   f(y) ≥ f(x) + ∇f(x)ᵀ(y − x) for all x, y   (tangent plane lies below the graph)
Second-order:  H(x) positive semidefinite for all x
(H(x) positive definite for all x ⇒ strictly convex)

Main results:
1. If f is convex, every local minimizer is a global minimizer.
2. If f is convex and differentiable, ∇f(x*) = 0 ⇔ x* is a global minimizer. The first-order necessary condition becomes sufficient.
3. If f is strictly convex, it has at most one global minimizer.

Common convex functions: ‖x‖², eˣ, |x|, quadratics ½xᵀQx − bᵀx with Q PSD, least squares ‖Ax − b‖² (Hessian 2AᵀA is PSD).

##Example 4: Chord and tangent pictures

f(x) = eˣ is convex: the chord is above the graph and the tangent is below it.
g(x) = x⁴ − 3x² + x is not convex: part of the graph lies above the chord, and part lies below the tangent.

In [ ]:
f_c = np.exp
df_c = np.exp
g_nc = lambda x: x**4 - 3*x**2 + x
dg_nc = lambda x: 4*x**3 - 6*x + 1

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, fn, dfn, (x1, x2), x0, name in [
        (ax[0], f_c, df_c, (-1.5, 1.5), 0.0, "eˣ (convex)"),
        (ax[1], g_nc, dg_nc, (-1.5, 1.3), 0.0, "x⁴ − 3x² + x (not convex)")]:
    xs = np.linspace(-2, 2, 300)
    a.plot(xs, fn(xs), lw=2, label="f")
    a.plot([x1, x2], [fn(x1), fn(x2)], "r--o", label="chord")
    a.plot(xs, fn(x0) + dfn(x0) * (xs - x0), "g:", lw=2, label=f"tangent at x = {x0}")
    a.set_ylim(fn(xs).min() - 1, min(fn(xs).max(), 8))
    a.set_title(name); a.legend()
plt.tight_layout(); plt.show()

##Example 5: Testing convexity numerically

Jensen test: sample random x, y, t and count how often f(tx + (1 − t)y) > t f(x) + (1 − t) f(y).
Hessian test: check the minimum Hessian eigenvalue at random points.

A violation proves f is not convex. Zero violations is evidence, not proof; the proof is the second-order condition shown in the notes.

In [ ]:
A = rng.normal(size=(20, 2))
b = rng.normal(size=20)

tests = {
    "‖x‖²":        (lambda v: v @ v,                     lambda v: 2 * np.eye(2)),
    "‖Ax − b‖²":   (lambda v: np.sum((A @ v - b)**2),    lambda v: 2 * A.T @ A),
    "Rosenbrock":  (rosen,                                rosen_hess),
}

for name, (fn, hess) in tests.items():
    violations, min_eig = 0, np.inf
    for _ in range(5000):
        x, y = rng.uniform(-2, 2, size=(2, 2))
        t = rng.uniform()
        if fn(t*x + (1-t)*y) > t*fn(x) + (1-t)*fn(y) + 1e-12:
            violations += 1
        min_eig = min(min_eig, np.linalg.eigvalsh(hess(x)).min())
    print(f"{name:12s} Jensen violations: {violations:5d}/5000   smallest Hessian eigenvalue: {min_eig:9.3f}")

##Example 6: Convex vs non-convex minimization

Convex quadratic with Q positive definite (so strictly convex): f(x) = ½ xᵀQx − bᵀx,   ∇f(x) = Qx − b = 0   →   x* = Q⁻¹b

By results 2 and 3, x* is the unique global minimizer, so a local optimizer reaches it from every start.

The non-convex g(x) = x⁴ − 3x² + x has two local minima. A local optimizer's answer depends on the starting point, and only one of them is global.

In [ ]:
Q = np.array([[3.0, 1.0], [1.0, 2.0]])
bq = np.array([1.0, -1.0])
fq = lambda v: 0.5 * v @ Q @ v - bq @ v

x_star = np.linalg.solve(Q, bq)
print("eigenvalues of Q:", np.round(np.linalg.eigvalsh(Q), 4))
print("x* = Q^-1 b     :", np.round(x_star, 6))
for _ in range(4):
    start = rng.uniform(-10, 10, size=2)
    print(f"start {np.round(start, 2)} -> {np.round(optimize.minimize(fq, start).x, 6)}")

print()
for s in [-2.0, -0.5, 0.5, 2.0]:
    xm = optimize.minimize(lambda x: g_nc(x[0]), [s]).x[0]
    print(f"g: start {s:5.1f} -> x = {xm:7.4f}, g(x) = {g_nc(xm):8.4f}, g''(x) = {12*xm**2 - 6:6.2f}")

xs = np.linspace(-2, 2, 300)
plt.plot(xs, g_nc(xs))
for xm in np.sort(np.roots([4, 0, -6, 1]).real)[[0, 2]]:
    plt.plot(xm, g_nc(xm), "o", ms=9)
plt.title("g(x) = x⁴ − 3x² + x: two local minima, one global"); plt.xlabel("x")
plt.show()

##3.3.3 Gradient Descent

Iteration, with step size (learning rate) α > 0: xₖ₊₁ = xₖ − α ∇f(xₖ)

Why it works: by the first-order Taylor expansion, f(xₖ₊₁) ≈ f(xₖ) − α‖∇f(xₖ)‖²

so each step lowers f when α is small and ∇f(xₖ) ≠ 0. GD stops moving only where ∇f = 0, so it finds critical points.

L-smooth: ‖∇f(x) − ∇f(y)‖ ≤ L‖x − y‖. For twice differentiable f, this means all Hessian eigenvalues are ≤ L.

Descent lemma: if f is L-smooth and α ≤ 1/L, f(xₖ₊₁) ≤ f(xₖ) − (α/2)‖∇f(xₖ)‖²

Convergence results:
- f L-smooth (not necessarily convex), α ≤ 1/L: f decreases every step and ∇f(xₖ) → 0.
- f convex and L-smooth, α = 1/L: f(xₖ) − f* ≤ L‖x₀ − x*‖² / (2k).
- f m-strongly convex (Hessian eigenvalues ≥ m > 0), α = 1/L: ‖xₖ − x*‖ ≤ (1 − m/L)ᵏ ‖x₀ − x*‖.
  The condition number κ = L/m controls the speed: larger κ means slower convergence.

For the quadratic f = ½xᵀQx − bᵀx: L = λmax(Q), m = λmin(Q), and GD converges if and only if 0 < α < 2/λmax.

In [ ]:
def gradient_descent(grad, x0, alpha, n_iter):
    path = [np.asarray(x0, float)]
    for _ in range(n_iter):
        path.append(path[-1] - alpha * grad(path[-1]))
    return np.array(path)

## Example 7: Effect of the step size

f(x) = ½ xᵀQx,   Q = diag(1, 10),   x* = (0, 0),   L = 10

So 1/L = 0.1 and 2/L = 0.2. We try α = 0.02 (too small), 0.1 (= 1/L), 0.19 (just under 2/L), and 0.21 (over 2/L).

In [ ]:
Q7 = np.diag([1.0, 10.0])
f7 = lambda v: 0.5 * v @ Q7 @ v
grad7 = lambda v: Q7 @ v
x0 = np.array([4.0, 1.5])
alphas = [0.02, 0.1, 0.19, 0.21]

xs = np.linspace(-5, 5, 200)
Xg, Yg = np.meshgrid(xs, xs)
Z = 0.5 * (Xg**2 + 10 * Yg**2)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].contour(Xg, Yg, Z, levels=20, cmap="Greys", alpha=0.6)
for a in alphas:
    path = gradient_descent(grad7, x0, a, 40)
    vals = np.array([f7(p) for p in path])
    print(f"alpha = {a:<5} f after 40 steps = {vals[-1]:.3e}")
    ax[0].plot(path[:15, 0], path[:15, 1], "o-", ms=3, label=f"α = {a}")
    ax[1].semilogy(vals, label=f"α = {a}")
ax[0].set_xlim(-5, 5); ax[0].set_ylim(-3, 3)
ax[0].set_title("First 15 GD steps"); ax[0].legend()
ax[1].set_xlabel("iteration"); ax[1].set_ylabel("f(xₖ) − f*")
ax[1].set_title("Convergence (log scale)"); ax[1].legend()
plt.tight_layout(); plt.show()

α = 0.02 converges slowly. α = 0.1 converges fast. α = 0.19 still converges but oscillates in the steep y-direction. α = 0.21 > 2/L diverges.

##Example 8: Checking the descent lemma and the convergence rate

Same f, with α = 1/L = 0.1. Here m = 1 and L = 10. Check two things:
-every step decreases f by at least (α/2)‖∇f(xₖ)‖²
-‖xₖ − x*‖ ≤ (1 − m/L)ᵏ ‖x₀ − x*‖

In [ ]:
L, m = 10.0, 1.0
alpha = 1 / L
path = gradient_descent(grad7, x0, alpha, 60)

decrease = np.array([f7(path[k]) - f7(path[k+1]) for k in range(60)])
bound = np.array([alpha / 2 * grad7(path[k]) @ grad7(path[k]) for k in range(60)])
print("descent lemma holds at every step:", np.all(decrease >= bound - 1e-15))

err = np.linalg.norm(path, axis=1)
rate_bound = (1 - m / L) ** np.arange(61) * err[0]
print("rate bound holds at every step:   ", np.all(err <= rate_bound + 1e-12))

plt.semilogy(err, label="‖xₖ − x*‖")
plt.semilogy(rate_bound, "--", label="(1 − m/L)ᵏ ‖x₀ − x*‖")
plt.xlabel("iteration"); plt.title("Linear convergence for a strongly convex f")
plt.legend(); plt.show()

##Example 9: Condition number

  Q = diag(1, κ),   α = 1/κ

Count the iterations needed to reach ‖xₖ − x*‖ < 10⁻⁶. Larger κ means the contours are long narrow ellipses, GD zigzags across the valley, and progress along the flat direction is slow.

In [ ]:
def iters_to_tol(kappa, tol=1e-6, max_iter=100000):
    Qk = np.diag([1.0, kappa])
    x = np.array([1.0, 1.0])
    for k in range(max_iter):
        if np.linalg.norm(x) < tol:
            return k
        x = x - (1 / kappa) * (Qk @ x)
    return max_iter

for kappa in [2, 10, 50, 200]:
    print(f"kappa = {kappa:4d}  iterations = {iters_to_tol(kappa)}")

Qz = np.diag([1.0, 20.0])
path = gradient_descent(lambda v: Qz @ v, [9.0, 1.0], 0.095, 30)
xs, ys = np.linspace(-10, 10, 200), np.linspace(-2, 2, 200)
Xg, Yg = np.meshgrid(xs, ys)
plt.contour(Xg, Yg, 0.5 * (Xg**2 + 20 * Yg**2), levels=20, cmap="Greys")
plt.plot(path[:, 0], path[:, 1], "ro-", ms=3)
plt.title("Zigzag on an ill-conditioned quadratic (κ = 20)")
plt.show()

##Example 10: Non-convex function

   g(x) = x⁴ − 3x² + x,   g'(x) = 4x³ − 6x + 1

GD converges to a critical point, but which one depends on x₀. Starting in the left basin reaches the global min. Starting in the right basin gets stuck at the local min. Starting exactly at a critical point never moves.

In [ ]:
alpha = 0.02
xs = np.linspace(-2, 2, 300)
plt.plot(xs, g_nc(xs), "k", lw=1)
crit = np.sort(np.roots([4, 0, -6, 1]).real)
print("critical points:", np.round(crit, 4))

for x0g in [-1.9, -0.3, 0.5, 1.9, crit[1]]:
    path = gradient_descent(lambda x: dg_nc(x), np.array([x0g]), alpha, 200).ravel()
    print(f"start {x0g:7.4f} -> x = {path[-1]:7.4f}, g = {g_nc(path[-1]):8.4f}")
    plt.plot(path, g_nc(path), "o-", ms=3, label=f"x₀ = {x0g:.2f}")
plt.title("GD on a non-convex function"); plt.xlabel("x"); plt.legend()
plt.show()

## Example 11: Least squares by gradient descent

Linear regression loss (convex, from Section 2.4):

  f(β) = (1/2n) ‖Xβ − y‖²
  ∇f(β) = (1/n) Xᵀ(Xβ − y)
  H = (1/n) XᵀX   (constant and PSD)   →   L = λmax(XᵀX / n)

GD with α = 1/L should reach the normal-equations solution β̂ = (XᵀX)⁻¹Xᵀy.

In [ ]:
n = 200
x = rng.uniform(0, 5, size=n)
X = np.column_stack([np.ones(n), x])
y = 3.0 - 2.0 * x + rng.normal(0, 1, size=n)

loss = lambda beta: np.sum((X @ beta - y)**2) / (2 * n)
grad_ls = lambda beta: X.T @ (X @ beta - y) / n

eig = np.linalg.eigvalsh(X.T @ X / n)
L_ls = eig.max()
print(f"Hessian eigenvalues: {np.round(eig, 4)}  ->  L = {L_ls:.4f}, kappa = {eig.max()/eig.min():.1f}")

path = gradient_descent(grad_ls, np.zeros(2), 1 / L_ls, 2000)
beta_gd = path[-1]
beta_ne = np.linalg.solve(X.T @ X, X.T @ y)
print("GD (2000 steps)  :", np.round(beta_gd, 6))
print("Normal equations :", np.round(beta_ne, 6))

losses = np.array([loss(b) for b in path])
plt.semilogy(losses - loss(beta_ne) + 1e-16)
plt.xlabel("iteration"); plt.ylabel("f(βₖ) − f(β̂)")
plt.title("GD on least squares")
plt.show()